#### Script to process CSV-file and load it into your normalized PostgreSQL database schema inside public.wildfires
The script handles:
1. Converting -9999 to NULL.
2. Keeping longitude/latitude separate.
3. Keeping split dates and creating a combined native DATE column (brand_date).
4. Creating lookup dimensional tables for bundesland, gemeinde (prioritizing gemeinde_neu), brandart, and ursache1.
5. Overwriting existing entries (UPSERT on conflict of objid).


In [4]:
pip install pandas sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 495.0 kB/s eta 0:00:05
   ------- -------------------------------- 0.5/2.8 MB 495.0 kB/s eta 0:00:05
   ----------- ---------------------------- 0.8/2.8 MB 610.5 kB/s eta 0:00:04
   --------------- ------------------------ 1.0/2.8 MB 739.9 kB/s eta 0:00:03
   ---------------------- ----------------- 1.6/2.8 MB 953.6 kB/s eta 0:00:02
   ------------------------------ --------- 2.1/2.8 MB 1.1 MB/s eta 0:00:01
   ------------------------------------- -- 2.6/2.8 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 

In [6]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

In [20]:
# PostgreSQL Database Connection Configuration
DB_USER = "postgres"
DB_PASSWORD = "Test1-"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "coders_end_db"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)


In [21]:
# Path to your raw CSV file
input_file_path = "../../data/teil_011-015.csv"  # "../data/boku_waldbrand_02-11-2026(teil_011-015).csv" # "../data/boku_waldbrand_02-11-2026.csv"
timewindow_days_per_case = 14

def load_and_clean_data(file_path):
    print("Reading and cleaning CSV file...")
    # Read dataset
    df = pd.read_csv(file_path)
    # Map -9999 to explicit NaN/Null across the entire dataframe
    df = df.replace([-9999, -9999.0, "-9999"], np.nan)
    
    # Save split dates AND create a combined native DATE column
    # Coercing errors to handle any malformed rows safely
    df['brand_date'] = pd.to_datetime(
        df[['jahr', 'monat', 'tag']].rename(columns={'jahr': 'year', 'monat': 'month', 'tag': 'day'}), 
        errors='coerce'
    ).dt.date

    # Standardise municipality: prioritize gemeinde_neu, fallback to gemeinde
    df['final_gemeinde'] = df['gemeinde_neu'].fillna(df['gemeinde']).str.strip()
    # Drop rows where municipality name cannot be determined
    df = df.dropna(subset=['final_gemeinde', 'bundesland'])    
    return df

def setup_database_schema(engine):
    print("Creating lookup dimension tables and core table...")
    with engine.begin() as conn:
        # Create Schema if it doesn't exist
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS public;"))
        # Dimension Table: bundesland
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS public.bundesland (
                code VARCHAR(10) PRIMARY KEY,
                name VARCHAR(100) NULL
            );
        """))
        
        # Dimension Table: gemeinde
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS public.gemeinde (
                gemeinde_id SERIAL PRIMARY KEY,
                name VARCHAR(255) NOT NULL,
                bundesland_code VARCHAR(10) REFERENCES public.bundesland(code),
                CONSTRAINT unique_gemeinde_per_state UNIQUE (name, bundesland_code)
            );
        """))
        
        # Table: brandart
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS public.brandart (
                brandart_id INT PRIMARY KEY,
                name VARCHAR(100) NULL
            );
        """))

        # Table: ursache
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS public.ursache (
                ursache_id INT PRIMARY KEY,
                name VARCHAR(100) NULL
            );
        """))
        
        # Fact/Core Table: wildfires
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS public.wildfires (
                objid INT PRIMARY KEY,
                waldbrandid BIGINT NULL,
                brand_date DATE NULL,
                jahr INT NULL,
                monat INT NULL,
                tag INT NULL,
                gemeinde_id INT REFERENCES public.gemeinde(gemeinde_id),
                brandart_id INT REFERENCES public.brandart(brandart_id),
                ursache_id INT REFERENCES public.ursache(ursache1_id),
                brandflaeche NUMERIC NULL,
                brandflaeche_korr NUMERIC NULL,
                feuerwehren INT NULL,
                einsatzkraefte INT NULL,
                quelle1 TEXT NULL,
                verifiziert BOOLEAN NULL,
                schutzwald SMALLINT NULL,
                longitude DOUBLE PRECISION NULL,
                latitude DOUBLE PRECISION NULL
            );
        """))

def populate_dimensions(df, engine):
    print("Populating lookup dimensional tables...")
    
    # Populate Bundesland Codes
    unique_states = df['bundesland'].dropna().unique()
    with engine.begin() as conn:
        for state in unique_states:
            conn.execute(
                text("INSERT INTO public.bundesland (code) VALUES (:code) ON CONFLICT (code) DO NOTHING;"),
                {"code": state}
            )
            
    # Populate Gemeinde (linked to Bundesland code)
    unique_gems = df[['final_gemeinde', 'bundesland']].drop_duplicates().dropna()
    with engine.begin() as conn:
        for _, row in unique_gems.iterrows():
            conn.execute(
                text("""
                    INSERT INTO public.gemeinde (name, bundesland_code) 
                    VALUES (:name, :code) 
                    ON CONFLICT (name, bundesland_code) DO NOTHING;
                """),
                {"name": row['final_gemeinde'], "code": row['bundesland']}
            )

    # Populate Brandart Codes dynamically from encountered IDs
    unique_brandart = df['brandart'].dropna().unique()
    with engine.begin() as conn:
        for b_id in unique_brandart:
            conn.execute(
                text("INSERT INTO public.brandart (brandart_id) VALUES (:id) ON CONFLICT (brandart_id) DO NOTHING;"),
                {"id": int(b_id)}
            )

    # Populate Ursache Codes dynamically from encountered IDs
    unique_ursache = df['ursache1'].dropna().unique()
    with engine.begin() as conn:
        for u_id in unique_ursache:
            conn.execute(
                text("INSERT INTO public.ursache (ursache_id) VALUES (:id) ON CONFLICT (ursache_id) DO NOTHING;"),
                {"id": int(u_id)}
            )

def load_core_wildfires(df, engine):
    print("Loading data into public.wildfires with UPSERT conflict handling...")
    
    # Fetch database generated IDs for municipalities to perform the mapping mapping
    with engine.connect() as conn:
        gem_lookup = pd.read_sql("SELECT gemeinde_id, name, bundesland_code FROM public.gemeinde", conn)
    
    # Merge dimension foreign keys back into main staging data frame
    df = df.merge(
        gem_lookup, 
        left_on=['final_gemeinde', 'bundesland'], 
        right_on=['name', 'bundesland_code'], 
        how='left'
    )
    
    # Build a clean database insert payload
    upsert_query = text("""
        INSERT INTO public.wildfires (
            objid, waldbrandid, brand_date, jahr, monat, tag, gemeinde_id, brandart_id, ursache1_id,
            brandflaeche, brandflaeche_korr, feuerwehren, einsatzkraefte, quelle1, verifiziert, 
            schutzwald, longitude, latitude
        ) VALUES (
            :objid, :waldbrandid, :brand_date, :jahr, :monat, :tag, :gemeinde_id, :brandart_id, :ursache1_id,
            :brandflaeche, :brandflaeche_korr, :feuerwehren, :einsatzkraefte, :quelle1, :verifiziert, 
            :schutzwald, :longitude, :latitude
        )
        ON CONFLICT (objid) DO UPDATE SET
            waldbrandid = EXCLUDED.waldbrandid,
            brand_date = EXCLUDED.brand_date,
            jahr = EXCLUDED.jahr,
            monat = EXCLUDED.monat,
            tag = EXCLUDED.tag,
            gemeinde_id = EXCLUDED.gemeinde_id,
            brandart_id = EXCLUDED.brandart_id,
            ursache1_id = EXCLUDED.ursache1_id,
            brandflaeche = EXCLUDED.brandflaeche,
            brandflaeche_korr = EXCLUDED.brandflaeche_korr,
            feuerwehren = EXCLUDED.feuerwehren,
            einsatzkraefte = EXCLUDED.einsatzkraefte,
            quelle1 = EXCLUDED.quelle1,
            verifiziert = EXCLUDED.verifiziert,
            schutzwald = EXCLUDED.schutzwald,
            longitude = EXCLUDED.longitude,
            latitude = EXCLUDED.latitude;
    """)
    
    # Safely convert data types for clean native insert via dictionary bindings
    with engine.begin() as conn:
        for _, r in df.iterrows():
            payload = {
                "objid": int(r['objid']),
                "waldbrandid": int(r['waldbrandid']) if pd.notna(r['waldbrandid']) else None,
                "brand_date": r['brand_date'] if pd.notna(r['brand_date']) else None,
                "jahr": int(r['jahr']) if pd.notna(r['jahr']) else None,
                "monat": int(r['monat']) if pd.notna(r['monat']) else None,
                "tag": int(r['tag']) if pd.notna(r['tag']) else None,
                "gemeinde_id": int(r['gemeinde_id']) if pd.notna(r['gemeinde_id']) else None,
                "brandart_id": int(r['brandart']) if pd.notna(r['brandart']) else None,
                "ursache1_id": int(r['ursache1']) if pd.notna(r['ursache1']) else None,
                "brandflaeche": float(r['brandflaeche']) if pd.notna(r['brandflaeche']) else None,
                "brandflaeche_korr": float(r['brandflaeche_korr']) if pd.notna(r['brandflaeche_korr']) else None,
                "feuerwehren": int(r['feuerwehren']) if pd.notna(r['feuerwehren']) else None,
                "einsatzkraefte": int(r['einsatzkraefte']) if pd.notna(r['einsatzkraefte']) else None,
                "quelle1": str(r['quelle1']) if pd.notna(r['quelle1']) else None,
                "verifiziert": bool(r['verifiziert']) if pd.notna(r['verifiziert']) else None,
                "schutzwald": int(r['schutzwald']) if pd.notna(r['schutzwald']) else None,
                "longitude": float(r['longitude']) if pd.notna(r['longitude']) else None,
                "latitude": float(r['latitude']) if pd.notna(r['latitude']) else None
            }
            conn.execute(upsert_query, payload)
            
    print("Database Pipeline finished successfully.")




In [23]:
# if __name__ == "__main__":
cleaned_df = load_and_clean_data(input_file_path)
# setup_database_schema(engine)
populate_dimensions(cleaned_df, engine)
load_core_wildfires(cleaned_df, engine)

Reading and cleaning CSV file...
Populating lookup dimensional tables...


OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"

(Background on this error at: https://sqlalche.me/e/20/e3q8)